In [ ]:
"""
=============================================================================
CICLO DE VIDA DE DATOS Y MODELO EN DEEP LEARNING
Integración completa con Weights & Biases (W&B)
=============================================================================
Requisitos:
    pip install wandb torch torchvision scikit-learn matplotlib seaborn

Uso:
    1. Autenticarse: wandb login
    2. Ejecutar: python deep_learning_pipeline.py
=============================================================================
"""

import os
import time
import json
import random
import logging
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

from sklearn.metrics import confusion_matrix, classification_report
import wandb

# =============================================================================
# CONFIGURACIÓN GLOBAL Y REPRODUCIBILIDAD
# =============================================================================

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S"
)
log = logging.getLogger(__name__)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
log.info(f"Dispositivo de cómputo: {DEVICE}")


# =============================================================================
# FASE 1 — DATA JOURNEY: PREPARACIÓN Y ACCESO A DATOS
# =============================================================================

class DataJourney:
    """
    Encapsula el ciclo completo de los datos:
    ingesta → transformación → validación → versionado → split → carga
    """

    # Estadísticas precalculadas de MNIST para normalización
    MEAN = (0.1307,)
    STD  = (0.3081,)

    def __init__(self, data_dir: str = "./data", batch_size: int = 64):
        self.data_dir   = Path(data_dir)
        self.batch_size = batch_size
        self.metadata   = {}

    # ------------------------------------------------------------------
    # 1.1 Definición de transformaciones (pipeline de preprocesamiento)
    # ------------------------------------------------------------------
    def build_transforms(self, augment: bool = True):
        """
        Construye pipelines de transformación diferenciados para
        entrenamiento (con augmentation) y evaluación (sin augmentation).
        """
        base = [
            transforms.ToTensor(),
            transforms.Normalize(self.MEAN, self.STD),
        ]
        if augment:
            train_tf = transforms.Compose([
                transforms.RandomRotation(10),
                transforms.RandomAffine(0, translate=(0.1, 0.1)),
                *base,
            ])
        else:
            train_tf = transforms.Compose(base)

        eval_tf = transforms.Compose(base)
        return train_tf, eval_tf

    # ------------------------------------------------------------------
    # 1.2 Descarga y carga del dataset
    # ------------------------------------------------------------------
    def load_raw(self, augment: bool = True):
        log.info("Descargando/verificando dataset MNIST …")
        train_tf, eval_tf = self.build_transforms(augment)

        full_train = datasets.MNIST(self.data_dir, train=True,
                                    download=True, transform=train_tf)
        test_ds    = datasets.MNIST(self.data_dir, train=False,
                                    download=True, transform=eval_tf)
        return full_train, test_ds

    # ------------------------------------------------------------------
    # 1.3 Validación de integridad (Data Quality Check)
    # ------------------------------------------------------------------
    def validate(self, dataset, name: str = "dataset"):
        log.info(f"Validando integridad de {name} …")
        assert len(dataset) > 0, "Dataset vacío"
        sample, label = dataset[0]
        assert sample.shape == (1, 28, 28), f"Shape inesperado: {sample.shape}"
        assert 0 <= label <= 9, f"Etiqueta fuera de rango: {label}"

        # Estadísticas básicas sobre una muestra aleatoria
        indices   = np.random.choice(len(dataset), min(1000, len(dataset)), replace=False)
        pixels    = torch.stack([dataset[i][0] for i in indices])
        self.metadata[name] = {
            "size":        len(dataset),
            "pixel_mean":  pixels.mean().item(),
            "pixel_std":   pixels.std().item(),
            "classes":     sorted(set(dataset.targets.tolist())),
            "num_classes": len(dataset.classes),
        }
        log.info(f"  ✓ {name}: {len(dataset):,} muestras — "
                 f"μ={self.metadata[name]['pixel_mean']:.4f}, "
                 f"σ={self.metadata[name]['pixel_std']:.4f}")
        return self.metadata[name]

    # ------------------------------------------------------------------
    # 1.4 Split estratificado train / validation
    # ------------------------------------------------------------------
    def split(self, full_train, val_ratio: float = 0.15):
        n_val   = int(len(full_train) * val_ratio)
        n_train = len(full_train) - n_val
        train_ds, val_ds = random_split(
            full_train, [n_train, n_val],
            generator=torch.Generator().manual_seed(SEED)
        )
        log.info(f"Split → train: {n_train:,} | val: {n_val:,}")
        return train_ds, val_ds

    # ------------------------------------------------------------------
    # 1.5 Creación de DataLoaders
    # ------------------------------------------------------------------
    def build_loaders(self, train_ds, val_ds, test_ds):
        kw = dict(num_workers=0, pin_memory=(DEVICE.type == "cuda"))
        loaders = {
            "train": DataLoader(train_ds, batch_size=self.batch_size,
                                shuffle=True,  **kw),
            "val":   DataLoader(val_ds,   batch_size=self.batch_size*2,
                                shuffle=False, **kw),
            "test":  DataLoader(test_ds,  batch_size=self.batch_size*2,
                                shuffle=False, **kw),
        }
        log.info(f"DataLoaders creados — batch_size={self.batch_size}")
        return loaders

    # ------------------------------------------------------------------
    # 1.6 Versionar datos como W&B Artifact
    # ------------------------------------------------------------------
    def log_artifact(self, run, artifact_name: str = "mnist-preprocessed"):
        artifact = wandb.Artifact(
            name        = artifact_name,
            type        = "dataset",
            description = "MNIST normalizado con data-augmentation (rotación + traslación)",
            metadata    = self.metadata,
        )
        # Registrar el directorio de datos
        artifact.add_dir(str(self.data_dir), name="raw")
        run.log_artifact(artifact)
        log.info(f"Artifact '{artifact_name}' registrado en W&B")
        return artifact


# =============================================================================
# FASE 2 — ARQUITECTURA DEL MODELO
# =============================================================================

class ConvNet(nn.Module):
    """
    CNN pequeña para clasificación MNIST.
    Arquitectura: Conv → BN → Pool → Conv → BN → Pool → FC → FC
    """

    def __init__(self, num_classes: int = 10, dropout: float = 0.3):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),   # 28×28 → 28×28
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),                   # → 14×14

            nn.Conv2d(32, 64, 3, padding=1),   # 14×14 → 14×14
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),                   # → 7×7

            nn.Conv2d(64, 128, 3, padding=1),  # 7×7 → 7×7
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 7 * 7, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


# =============================================================================
# FASE 3 — ENTRENAMIENTO CON MONITOREO Y LOGGING
# =============================================================================

class Trainer:
    """
    Entrena el modelo con seguimiento completo en W&B:
    - métricas por epoch (loss, accuracy)
    - gradientes y pesos (wandb.watch)
    - checkpoint del mejor modelo
    - visualizaciones intermedias
    """

    def __init__(self, model, loaders, cfg, run):
        self.model   = model.to(DEVICE)
        self.loaders = loaders
        self.cfg     = cfg
        self.run     = run

        self.criterion = nn.CrossEntropyLoss()
        self.optimizer = optim.AdamW(
            model.parameters(),
            lr           = cfg["lr"],
            weight_decay = cfg["weight_decay"],
        )
        self.scheduler = optim.lr_scheduler.CosineAnnealingLR(
            self.optimizer, T_max=cfg["epochs"]
        )
        self.best_val_acc = 0.0
        self.history      = {"train_loss": [], "val_loss": [],
                              "train_acc":  [], "val_acc":  []}

    # ------------------------------------------------------------------
    def _run_epoch(self, split: str):
        training = (split == "train")
        self.model.train(training)
        loader = self.loaders[split]

        total_loss, correct, total = 0.0, 0, 0
        with torch.set_grad_enabled(training):
            for xb, yb in loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                logits = self.model(xb)
                loss   = self.criterion(logits, yb)

                if training:
                    self.optimizer.zero_grad()
                    loss.backward()
                    # Gradient clipping para estabilidad
                    nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                    self.optimizer.step()

                total_loss += loss.item() * xb.size(0)
                correct    += (logits.argmax(1) == yb).sum().item()
                total      += xb.size(0)

        return total_loss / total, correct / total

    # ------------------------------------------------------------------
    def train(self):
        # Monitorear gradientes y pesos en W&B
        wandb.watch(self.model, log="all", log_freq=100)

        log.info("=" * 60)
        log.info(f"Iniciando entrenamiento — {self.cfg['epochs']} epochs")
        log.info("=" * 60)

        for epoch in range(1, self.cfg["epochs"] + 1):
            t0 = time.time()

            train_loss, train_acc = self._run_epoch("train")
            val_loss,   val_acc   = self._run_epoch("val")
            self.scheduler.step()

            elapsed = time.time() - t0
            lr_now  = self.scheduler.get_last_lr()[0]

            # ---- Registro en W&B ----
            self.run.log({
                "epoch":      epoch,
                "train/loss": train_loss,
                "train/acc":  train_acc,
                "val/loss":   val_loss,
                "val/acc":    val_acc,
                "lr":         lr_now,
                "epoch_time": elapsed,
            }, step=epoch)

            # ---- Historial local ----
            self.history["train_loss"].append(train_loss)
            self.history["val_loss"].append(val_loss)
            self.history["train_acc"].append(train_acc)
            self.history["val_acc"].append(val_acc)

            log.info(
                f"Epoch {epoch:02d}/{self.cfg['epochs']} | "
                f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
                f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} | "
                f"lr={lr_now:.6f} | {elapsed:.1f}s"
            )

            # ---- Guardar mejor checkpoint ----
            if val_acc > self.best_val_acc:
                self.best_val_acc = val_acc
                ckpt_path = "best_model.pt"
                torch.save({
                    "epoch":     epoch,
                    "model":     self.model.state_dict(),
                    "optimizer": self.optimizer.state_dict(),
                    "val_acc":   val_acc,
                }, ckpt_path)
                self.run.summary["best_val_acc"]   = val_acc
                self.run.summary["best_epoch"]     = epoch
                log.info(f"  ★ Nuevo mejor modelo guardado (val_acc={val_acc:.4f})")

        log.info("Entrenamiento finalizado.")
        return self.history


# =============================================================================
# FASE 4 — EVALUACIÓN Y VISUALIZACIONES
# =============================================================================

class Evaluator:

    def __init__(self, model, loader, run, class_names):
        self.model       = model
        self.loader      = loader
        self.run         = run
        self.class_names = class_names

    def evaluate(self):
        self.model.eval()
        all_preds, all_labels, all_probs = [], [], []

        with torch.no_grad():
            for xb, yb in self.loader:
                xb = xb.to(DEVICE)
                logits = self.model(xb)
                probs  = F.softmax(logits, dim=1)
                preds  = logits.argmax(1)
                all_preds.extend(preds.cpu().tolist())
                all_labels.extend(yb.tolist())
                all_probs.extend(probs.cpu().tolist())

        acc = sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels)
        log.info(f"Test accuracy: {acc:.4f}")

        # ---- Matriz de confusión → W&B ----
        cm = confusion_matrix(all_labels, all_preds)
        fig, ax = plt.subplots(figsize=(10, 8))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                    xticklabels=self.class_names,
                    yticklabels=self.class_names, ax=ax)
        ax.set_title("Matriz de Confusión — Test Set", fontsize=14, fontweight="bold")
        ax.set_xlabel("Predicho"); ax.set_ylabel("Real")
        plt.tight_layout()
        self.run.log({"confusion_matrix": wandb.Image(fig)})
        plt.close(fig)

        # ---- Tabla W&B por clase ----
        report = classification_report(all_labels, all_preds,
                                       target_names=self.class_names,
                                       output_dict=True)
        table = wandb.Table(columns=["clase", "precision", "recall", "f1", "support"])
        for cls in self.class_names:
            r = report[cls]
            table.add_data(cls, round(r["precision"],4),
                           round(r["recall"],4), round(r["f1-score"],4),
                           int(r["support"]))
        self.run.log({"metricas_por_clase": table})

        # ---- Log final ----
        self.run.log({"test/accuracy": acc})
        self.run.summary["test_accuracy"] = acc
        return acc, all_preds, all_labels, all_probs

    def log_sample_predictions(self, dataset, n: int = 16):
        """Visualiza n predicciones del test set en W&B."""
        indices = random.sample(range(len(dataset)), n)
        images  = []
        self.model.eval()
        with torch.no_grad():
            for idx in indices:
                img, true_label = dataset[idx]
                logits = self.model(img.unsqueeze(0).to(DEVICE))
                pred   = logits.argmax(1).item()
                conf   = F.softmax(logits, dim=1).max().item()
                caption = (f"Real: {self.class_names[true_label]} | "
                           f"Pred: {self.class_names[pred]} ({conf:.2%})")
                images.append(wandb.Image(img, caption=caption))
        self.run.log({"predicciones_muestra": images})


# =============================================================================
# FASE 5 — MODEL SERVING (demo local con FastAPI-style pseudocódigo)
# =============================================================================

class ModelServingDemo:
    """
    Demuestra los patrones de Model Serving:
    - Exportación a ONNX / TorchScript
    - Clase de inferencia lista para empaquetarse en un microservicio
    """

    def __init__(self, model, run):
        self.model = model
        self.run   = run

    def export_torchscript(self, path: str = "model_scripted.pt"):
        self.model.eval()
        dummy   = torch.zeros(1, 1, 28, 28).to(DEVICE)
        scripted = torch.jit.trace(self.model, dummy)
        torch.jit.save(scripted, path)
        log.info(f"TorchScript guardado → {path}")
        artifact = wandb.Artifact("model-torchscript", type="model",
                                  description="Modelo exportado con TorchScript para serving")
        artifact.add_file(path)
        self.run.log_artifact(artifact)

    def export_onnx(self, path: str = "model.onnx"):
        self.model.eval()
        dummy = torch.zeros(1, 1, 28, 28).to(DEVICE)
        torch.onnx.export(
            self.model, dummy, path,
            input_names=["image"], output_names=["logits"],
            dynamic_axes={"image": {0: "batch"}, "logits": {0: "batch"}},
            opset_version=17,
        )
        log.info(f"ONNX exportado → {path}")
        artifact = wandb.Artifact("model-onnx", type="model",
                                  description="Modelo exportado en formato ONNX")
        artifact.add_file(path)
        self.run.log_artifact(artifact)

    def simulate_inference(self, loader, n_requests: int = 50):
        """Simula latencia de inferencia en producción."""
        self.model.eval()
        latencies = []
        images, _ = next(iter(loader))
        images = images[:n_requests].to(DEVICE)

        with torch.no_grad():
            for i in range(n_requests):
                t0  = time.perf_counter()
                _   = self.model(images[i:i+1])
                lat = (time.perf_counter() - t0) * 1000
                latencies.append(lat)
                self.run.log({
                    "serving/latency_ms": lat,
                    "serving/request_id": i + 1,
                })

        stats = {
            "serving/p50_ms":  np.percentile(latencies, 50),
            "serving/p95_ms":  np.percentile(latencies, 95),
            "serving/p99_ms":  np.percentile(latencies, 99),
            "serving/mean_ms": np.mean(latencies),
        }
        self.run.log(stats)
        log.info(f"Serving simulation — p50={stats['serving/p50_ms']:.2f}ms "
                 f"p95={stats['serving/p95_ms']:.2f}ms "
                 f"p99={stats['serving/p99_ms']:.2f}ms")
        return stats


# =============================================================================
# PIPELINE PRINCIPAL
# =============================================================================

def main():
    # ---- Hiperparámetros (sweep-friendly) ----
    config = {
        "project":      "ciclo-vida-deep-learning",
        "entity":       None,           # reemplazar con tu usuario/org W&B
        "model":        "ConvNet",
        "dataset":      "MNIST",
        "batch_size":   64,
        "epochs":       10,
        "lr":           1e-3,
        "weight_decay": 1e-4,
        "dropout":      0.3,
        "val_ratio":    0.15,
        "augment":      True,
        "seed":         SEED,
    }

    # ================================================================
    # INICIO DE RUN EN W&B
    # ================================================================
    run = wandb.init(
        project = config["project"],
        config  = config,
        tags    = ["mnist", "convnet", "data-journey"],
        notes   = "Actividad cooperativa — ciclo de vida datos y modelo",
    )
    cfg = wandb.config   # permite sweeps

    # ================================================================
    # FASE 1 — DATA JOURNEY
    # ================================================================
    log.info("\n" + "="*60)
    log.info("FASE 1 — DATA JOURNEY")
    log.info("="*60)

    journey = DataJourney(batch_size=cfg.batch_size)
    full_train, test_ds = journey.load_raw(augment=cfg.augment)

    journey.validate(full_train, "train_full")
    journey.validate(test_ds,    "test")

    train_ds, val_ds = journey.split(full_train, val_ratio=cfg.val_ratio)
    loaders = journey.build_loaders(train_ds, val_ds, test_ds)

    # Log estadísticas de datos en W&B
    run.log({
        "data/train_size": len(train_ds),
        "data/val_size":   len(val_ds),
        "data/test_size":  len(test_ds),
    })
    # Guardar artifact de datos
    journey.log_artifact(run)

    # ================================================================
    # FASE 2 — MODELO
    # ================================================================
    log.info("\n" + "="*60)
    log.info("FASE 2 — CONSTRUCCIÓN DEL MODELO")
    log.info("="*60)

    model = ConvNet(dropout=cfg.dropout)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    log.info(f"Parámetros entrenables: {n_params:,}")
    run.log({"model/trainable_params": n_params})

    # ================================================================
    # FASE 3 — ENTRENAMIENTO CON MONITOREO
    # ================================================================
    log.info("\n" + "="*60)
    log.info("FASE 3 — ENTRENAMIENTO Y MONITOREO")
    log.info("="*60)

    trainer = Trainer(model, loaders, dict(cfg), run)
    history = trainer.train()

    # ================================================================
    # FASE 4 — EVALUACIÓN
    # ================================================================
    log.info("\n" + "="*60)
    log.info("FASE 4 — EVALUACIÓN")
    log.info("="*60)

    # Cargar mejor checkpoint
    ckpt = torch.load("best_model.pt", map_location=DEVICE, weights_only=True)
    model.load_state_dict(ckpt["model"])
    log.info(f"Checkpoint cargado — epoch {ckpt['epoch']}, val_acc={ckpt['val_acc']:.4f}")

    class_names = [str(i) for i in range(10)]
    evaluator = Evaluator(model, loaders["test"], run, class_names)
    test_acc, _, _, _ = evaluator.evaluate()
    evaluator.log_sample_predictions(test_ds)

    # ================================================================
    # FASE 5 — MODEL SERVING
    # ================================================================
    log.info("\n" + "="*60)
    log.info("FASE 5 — MODEL SERVING")
    log.info("="*60)

    serving = ModelServingDemo(model, run)
    serving.export_torchscript()
    serving.export_onnx()
    serving.simulate_inference(loaders["test"])

    # ================================================================
    # LOG FINAL DE ARTEFACTO DE MODELO ENTRENADO
    # ================================================================
    model_artifact = wandb.Artifact(
        name        = "convnet-mnist-trained",
        type        = "model",
        description = f"ConvNet entrenada en MNIST — test_acc={test_acc:.4f}",
        metadata    = {
            "test_accuracy":  test_acc,
            "best_val_acc":   trainer.best_val_acc,
            "epochs_trained": cfg.epochs,
            "params":         n_params,
        },
    )
    model_artifact.add_file("best_model.pt")
    run.log_artifact(model_artifact)

    run.finish()
    log.info("\n✅ Pipeline completo. Revisa tu proyecto en https://wandb.ai")


if __name__ == "__main__":
    main()


data/test_size,▁
data/train_size,▁
data/val_size,▁
epoch,▁▂▃▄▅▅▆▇█
epoch_time,▂█▄▂▂▂▂▁▂
lr,█▇▆▅▄▃▂▁▁
model/trainable_params,▁
test/accuracy,▁
train/acc,▁▃▄▅▆▇▇██
train/loss,█▆▅▄▃▂▂▁▁
+2,...


wandb: Adding directory to artifact (data)... Done. 0.5s
wandb: WARNING Artifact "mnist-preprocessed" already exists with the same content. No new version will be created.
